# Escalado de features

El **escalado** transforma las features numéricas para que tengan rangos comparables. Muchos algoritmos de Machine Learning son sensibles a la escala de los datos y funcionan mejor cuando las features están normalizadas.

## Por qué escalar

Las features con diferentes escalas pueden causar problemas:

In [1]:
import numpy as np
import pandas as pd

# Features con escalas muy diferentes
datos = pd.DataFrame({
    'edad': [25, 30, 35, 40, 45],           # Rango: 25-45
    'salario': [30000, 45000, 50000, 60000, 75000],  # Rango: 30000-75000
    'habitaciones': [1, 2, 3, 3, 4]         # Rango: 1-4
})

print(datos.describe().loc[['min', 'max', 'std']])

          edad       salario  habitaciones
min  25.000000  30000.000000      1.000000
max  45.000000  75000.000000      4.000000
std   7.905694  16807.736314      1.140175


> El salario domina numéricamente sobre las otras features. Algoritmos como KNN o regresión logística pueden dar peso excesivo a features con valores grandes.

### Algoritmos sensibles a la escala

| Sensible a escala | No sensible a escala |
|-------------------|---------------------|
| KNN | Árboles de decisión |
| Regresión logística | Random Forest |
| SVM | Gradient Boosting |
| Redes neuronales | |

## StandardScaler

Transforma los datos para que tengan **media 0** y **desviación típica 1**:

$$z = \frac{x - \mu}{\sigma}$$

In [2]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Datos de ejemplo
X = np.array([[25, 30000], [30, 45000], [35, 50000], [40, 60000], [45, 75000]])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos originales:")
print(X)
print("\nDatos escalados:")
print(X_scaled.round(3))
print(f"\nMedia después de escalar: {X_scaled.mean(axis=0).round(6)}")
print(f"Std después de escalar: {X_scaled.std(axis=0).round(3)}")

Datos originales:
[[   25 30000]
 [   30 45000]
 [   35 50000]
 [   40 60000]
 [   45 75000]]

Datos escalados:
[[-1.414 -1.463]
 [-0.707 -0.466]
 [ 0.    -0.133]
 [ 0.707  0.532]
 [ 1.414  1.53 ]]

Media después de escalar: [0. 0.]
Std después de escalar: [1. 1.]


### Atributos aprendidos

In [3]:
print(f"Media original: {scaler.mean_}")
print(f"Desviación típica: {scaler.scale_}")

Media original: [3.5e+01 5.2e+04]
Desviación típica: [7.07106781e+00 1.50332964e+04]


> StandardScaler es la opción por defecto para la mayoría de casos. Funciona bien cuando los datos siguen una distribución aproximadamente normal.

## MinMaxScaler

Transforma los datos a un **rango específico**, por defecto [0, 1]:

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [4]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(X)

print("Datos escalados a [0, 1]:")
print(X_minmax.round(3))
print(f"\nMínimo: {X_minmax.min(axis=0)}")
print(f"Máximo: {X_minmax.max(axis=0)}")

Datos escalados a [0, 1]:
[[0.    0.   ]
 [0.25  0.333]
 [0.5   0.444]
 [0.75  0.667]
 [1.    1.   ]]

Mínimo: [0. 0.]
Máximo: [1. 1.]


### Rango personalizado

In [5]:
# Escalar al rango [-1, 1]
minmax_custom = MinMaxScaler(feature_range=(-1, 1))
X_custom = minmax_custom.fit_transform(X)

print("Datos escalados a [-1, 1]:")
print(X_custom.round(3))

Datos escalados a [-1, 1]:
[[-1.    -1.   ]
 [-0.5   -0.333]
 [ 0.    -0.111]
 [ 0.5    0.333]
 [ 1.     1.   ]]


> MinMaxScaler es útil cuando necesitas valores en un rango específico, por ejemplo para redes neuronales que esperan entradas entre 0 y 1.

## RobustScaler

Usa la **mediana** y el **rango intercuartílico (IQR)** en lugar de media y desviación típica. Es robusto frente a **outliers**:

$$x_{scaled} = \frac{x - mediana}{IQR}$$

In [6]:
from sklearn.preprocessing import RobustScaler

# Datos con outliers
X_outliers = np.array([[25, 30000], [30, 45000], [35, 50000], [40, 60000], [200, 500000]])

# Comparar StandardScaler vs RobustScaler
standard = StandardScaler()
robust = RobustScaler()

X_standard = standard.fit_transform(X_outliers)
X_robust = robust.fit_transform(X_outliers)

print("Con StandardScaler (afectado por outliers):")
print(X_standard.round(3))

print("\nCon RobustScaler (robusto a outliers):")
print(X_robust.round(3))

Con StandardScaler (afectado por outliers):
[[-0.61  -0.589]
 [-0.536 -0.506]
 [-0.461 -0.479]
 [-0.387 -0.424]
 [ 1.994  1.997]]

Con RobustScaler (robusto a outliers):
[[-1.    -1.333]
 [-0.5   -0.333]
 [ 0.     0.   ]
 [ 0.5    0.667]
 [16.5   30.   ]]


```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    subgraph StandardScaler
        S1[Usa media y std]
        S2[Sensible a outliers]
    end
    
    subgraph RobustScaler
        R1[Usa mediana e IQR]
        R2[Robusto a outliers]
    end
    
    style S2 fill:#f56565,stroke:#c53030,color:#fff
    style R2 fill:#48bb78,stroke:#276749,color:#fff
```

> Usa RobustScaler cuando tus datos contengan outliers que no quieras eliminar pero tampoco quieras que dominen el escalado.

## MaxAbsScaler

Escala dividiendo por el **valor absoluto máximo**. Mantiene el cero y no centra los datos:

In [7]:
from sklearn.preprocessing import MaxAbsScaler

maxabs = MaxAbsScaler()
X_maxabs = maxabs.fit_transform(X)

print("Datos escalados por valor máximo absoluto:")
print(X_maxabs.round(3))

Datos escalados por valor máximo absoluto:
[[0.556 0.4  ]
 [0.667 0.6  ]
 [0.778 0.667]
 [0.889 0.8  ]
 [1.    1.   ]]


> MaxAbsScaler es útil para datos dispersos (sparse) porque no destruye la estructura de ceros.

## Normalizer

A diferencia de los otros escaladores, `Normalizer` escala cada **fila** (muestra) para que tenga norma unitaria:

In [8]:
from sklearn.preprocessing import Normalizer

# Normalizar por filas (cada muestra tiene norma 1)
normalizer = Normalizer(norm='l2')
X_normalized = normalizer.fit_transform(X)

print("Datos normalizados (norma L2 por fila):")
print(X_normalized.round(6))

# Verificar que cada fila tiene norma 1
normas = np.linalg.norm(X_normalized, axis=1)
print(f"\nNorma de cada fila: {normas.round(3)}")

Datos normalizados (norma L2 por fila):
[[8.33e-04 1.00e+00]
 [6.67e-04 1.00e+00]
 [7.00e-04 1.00e+00]
 [6.67e-04 1.00e+00]
 [6.00e-04 1.00e+00]]

Norma de cada fila: [1. 1. 1. 1. 1.]


> Normalizer se usa en casos específicos como procesamiento de texto (TF-IDF) donde importa la dirección del vector, no su magnitud.

## Comparación de escaladores

In [9]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import numpy as np

# Datos con diferentes características
np.random.seed(42)
X_normal = np.random.randn(100, 2) * 10 + 50  # Distribución normal
X_normal = np.vstack([X_normal, [[200, 300]]])  # Añadir outlier

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

print("Estadísticas después de escalar (con outlier):\n")
for nombre, scaler in scalers.items():
    X_scaled = scaler.fit_transform(X_normal)
    print(f"{nombre}:")
    print(f"  Min: {X_scaled.min(axis=0).round(2)}")
    print(f"  Max: {X_scaled.max(axis=0).round(2)}")
    print(f"  Media: {X_scaled.mean(axis=0).round(2)}")
    print()

Estadísticas después de escalar (con outlier):

StandardScaler:
  Min: [-1.54 -0.85]
  Max: [8.7  9.28]
  Media: [0. 0.]

MinMaxScaler:
  Min: [0. 0.]
  Max: [1. 1.]
  Media: [0.15 0.08]

RobustScaler:
  Min: [-2.43 -1.58]
  Max: [14.14 18.28]
  Media: [0.06 0.09]



### Cuándo usar cada escalador

| Escalador | Cuándo usar |
|-----------|-------------|
| StandardScaler | Opción por defecto, datos sin outliers extremos |
| MinMaxScaler | Necesitas rango específico, sin outliers |
| RobustScaler | Datos con outliers |
| MaxAbsScaler | Datos dispersos (sparse) |
| Normalizer | Normalización por muestra (textos, vectores) |

## Escalado en pipelines

Siempre usa escaladores dentro de un **Pipeline** para evitar data leakage:

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer

# Cargar datos
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Pipeline con escalado
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', KNeighborsClassifier(n_neighbors=5))
])

# Validación cruzada
scores = cross_val_score(pipeline, X, y, cv=5)
print(f"KNN con escalado: {scores.mean():.4f} (+/- {scores.std():.4f})")

KNN con escalado: 0.9649 (+/- 0.0096)


### Comparar con y sin escalado

In [11]:
# Sin escalado
knn_sin_escalar = KNeighborsClassifier(n_neighbors=5)
scores_sin = cross_val_score(knn_sin_escalar, X, y, cv=5)

# Con escalado
pipeline_con = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', KNeighborsClassifier(n_neighbors=5))
])
scores_con = cross_val_score(pipeline_con, X, y, cv=5)

print(f"KNN sin escalar: {scores_sin.mean():.4f} (+/- {scores_sin.std():.4f})")
print(f"KNN con escalar: {scores_con.mean():.4f} (+/- {scores_con.std():.4f})")

KNN sin escalar: 0.9279 (+/- 0.0218)
KNN con escalar: 0.9649 (+/- 0.0096)


> El escalado mejora significativamente el rendimiento de KNN porque este algoritmo usa distancias euclidianas.

## Comparar diferentes escaladores

In [12]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

print("Comparación de escaladores con KNN:\n")
for nombre, scaler in scalers.items():
    pipeline = Pipeline([
        ('scaler', scaler),
        ('modelo', KNeighborsClassifier(n_neighbors=5))
    ])
    scores = cross_val_score(pipeline, X, y, cv=5)
    print(f"{nombre}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Comparación de escaladores con KNN:



StandardScaler: 0.9649 (+/- 0.0096)
MinMaxScaler: 0.9719 (+/- 0.0116)


RobustScaler: 0.9631 (+/- 0.0129)


## Parámetros importantes

### StandardScaler

In [13]:
# Controlar si centrar y/o escalar
scaler_no_center = StandardScaler(with_mean=False)  # No restar media
scaler_no_scale = StandardScaler(with_std=False)    # No dividir por std

### MinMaxScaler

In [14]:
# Rango personalizado
scaler_rango = MinMaxScaler(feature_range=(0, 100))

# clip: forzar nuevos datos al rango
scaler_clip = MinMaxScaler(clip=True)

## Buenas prácticas

### Lista de verificación

| Verificación | Descripción |
|--------------|-------------|
| Escalar dentro de Pipeline | Evita data leakage |
| Elegir escalador apropiado | Según características de los datos |
| Probar sin escalar primero | Algunos modelos no lo necesitan |
| Verificar el impacto | Comparar métricas con y sin escalado |

### Orden en el pipeline

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    A[Datos] --> B[Imputar]
    B --> C[Escalar]
    C --> D[Modelo]
    
    style B fill:#4a90d9,stroke:#2c5282,color:#fff
    style C fill:#4a90d9,stroke:#2c5282,color:#fff
    style D fill:#48bb78,stroke:#276749,color:#fff
```

* **Imputar primero**: los escaladores no funcionan bien con NaN
* **Escalar después**: una vez los datos están completos

El escalado es un paso fundamental del preprocesamiento para algoritmos basados en distancias o gradientes. La elección del escalador depende de las características de los datos, especialmente la presencia de outliers.